# Stage 5 - Topic modeling

Find the main themes in the comments. Two methods, same as Lab 4:
- **BERTopic** (embeddings + clustering) - main one, gives nice readable topics
- **LDA** (sklearn) - classic baseline to compare against

Then we redo BERTopic *per sentiment* (positive vs negative) which is one of the deliverables -
what are people happy about vs complaining about.

In: `../data/comments_sentiment.csv`  Out: `../data/comments_topics.csv`

In [2]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

c:\Users\Saeed\Documents\370\notebooks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("../data/comments_sentiment.csv").dropna(subset=["clean_text"]).reset_index(drop=True)
docs = df["clean_text"].tolist()
print(len(docs), "documents")

18771 documents


## BERTopic - overall topics

In [4]:
# stopword-aware vectorizer so topic words aren't 'the', 'and' ...
vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2))

topic_model = BERTopic(vectorizer_model=vectorizer, min_topic_size=50, verbose=True)
topics, probs = topic_model.fit_transform(docs)
df["topic"] = topics

topic_model.get_topic_info().head(15)

2026-06-07 02:08:57,878 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 587/587 [01:39<00:00,  5.87it/s]
2026-06-07 02:10:43,277 - BERTopic - Embedding - Completed ✓
2026-06-07 02:10:43,278 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-07 02:11:11,114 - BERTopic - Dimensionality - Completed ✓
2026-06-07 02:11:11,116 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-07 02:11:14,717 - BERTopic - Cluster - Completed ✓
2026-06-07 02:11:14,723 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-07 02:11:15,293 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,6446,-1_phone_video_brother_16,"[phone, video, brother, 16, phone 16, professi...",[عوزين مقارنه بين phone 15 professional max an...
1,0,1010,0_ofof_60_ofof ofof_16,"[ofof, 60, ofof ofof, 16, 15, 120, 16 16, 16 1...","[شاشة 60 هرتس وكأنك تطبل للايفون, و أنت عندك 1..."
2,1,986,1_phone 16_16_phone_phone 15,"[phone 16, 16, phone, phone 15, 15, profession...","[phone 16, i phone 16, phone 16]"
3,2,781,2_60hz_120hz_60_refresh,"[60hz, 120hz, 60, refresh, 120, display, rate,...","[60hz??? no., it’s still 60hz, 60hz]"
4,3,677,3_hello_iss_sick iss_sick,"[hello, iss, sick iss, sick, ke, hi, ni, nasi,...","[that on on hello, phone Sick Iss all, that a ..."
5,4,665,4_camera_button_camera button_phone,"[camera, button, camera button, phone, control...","[where’s the third camera?, the camera, camera]"
6,5,445,5_phone phone_phone_just_new phone,"[phone phone, phone, just, new phone, buy, new...","[phone, i what phone can you give now me, can ..."
7,6,436,6_anna_anna anna_lo_anna phone,"[anna, anna anna, lo, anna phone, ahead, led, ...","[his anna, anna, anna]"
8,7,385,7_apple_intelligence_artificial intelligence_a...,"[apple, intelligence, artificial intelligence,...","[apple, apple intelligence, Artificial Intelli..."
9,8,370,8_nice game_sa_game_applicable,"[nice game, sa, game, applicable, nice, long, ...",[per Parents Are One pinagcompare mo maman you...


In [5]:
# words for the biggest few topics
for t in topic_model.get_topic_info()["Topic"].head(6):
    if t == -1:
        continue  # -1 is the outlier/junk bucket
    words = ", ".join(w for w, _ in topic_model.get_topic(t)[:8])
    print(f"Topic {t}: {words}")

Topic 0: ofof, 60, ofof ofof, 16, 15, 120, 16 16, 16 15
Topic 1: phone 16, 16, phone, phone 15, 15, professional, 16 phone, 17
Topic 2: 60hz, 120hz, 60, refresh, 120, display, rate, refresh rate
Topic 3: hello, iss, sick iss, sick, ke, hi, ni, nasi
Topic 4: camera, button, camera button, phone, control, photo, camera phone, camera control


In [6]:
# BERTopic has nice built in plots (works in notebook)
topic_model.visualize_barchart(top_n_topics=8)

## LDA baseline

Classic LDA on a bag-of-words for comparison.

In [7]:
cv = CountVectorizer(stop_words="english", max_features=2000, min_df=10)
dtm = cv.fit_transform(df["lemmas"].fillna(""))

lda = LatentDirichletAllocation(n_components=8, random_state=0)
lda.fit(dtm)

words = cv.get_feature_names_out()
for i, comp in enumerate(lda.components_):
    top = [words[j] for j in comp.argsort()[::-1][:8]]
    print(f"LDA topic {i}: {', '.join(top)}")

LDA topic 0: teach, انا, برو, ايفون, boss, على, انت, ولا
LDA topic 1: phone, professional, max, battery, year, model, want, upgrade
LDA topic 2: hello, phone, sick, iss, yeah, hold, like, sir
LDA topic 3: samson, phone, nice, 60hz, game, apple, applicable, long
LDA topic 4: phone, brother, watch, plus, buy, review, professional, ask
LDA topic 5: anna, camera, review, button, phone, motor, video, undo
LDA topic 6: apple, new, god, money, comment, happy, update, right
LDA topic 7: phone, like, video, good, look, love, brother, thank


## Topics per sentiment

One of the deliverables - run topic modeling separately on positive and negative comments so we can
say *what people like* vs *what they complain about*.

In [8]:
def topics_for(sentiment, min_size=30):
    sub = df[df["sentiment"] == sentiment]["clean_text"].tolist()
    if len(sub) < min_size * 2:
        print("not enough", sentiment, "comments")
        return None
    m = BERTopic(vectorizer_model=CountVectorizer(stop_words="english"),
                 min_topic_size=min_size)
    t, _ = m.fit_transform(sub)
    print(f"=== {sentiment.upper()} ===")
    for tid in m.get_topic_info()["Topic"].head(6):
        if tid == -1:
            continue
        print(f"  topic {tid}:", ", ".join(w for w, _ in m.get_topic(tid)[:6]))
    return m

pos_model = topics_for("positive")
neg_model = topics_for("negative")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5721.88it/s]


=== POSITIVE ===
  topic 0: video, thanks, thank, brother, review, watching
  topic 1: 16, professional, 15, 17, upgrade, better
  topic 2: 60hz, 120hz, 120, 60, refresh, display
  topic 3: hello, iss, sick, ke, hi, ni
  topic 4: samson, galaxy, ultra, better, best, phone


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4478.35it/s]


=== NEGATIVE ===
  topic 0: pass, hello, sa, anna, applicable, nice
  topic 1: 16, phone, professional, 15, max, upgrade
  topic 2: 60hz, 120hz, professional, 60, refresh, phone
  topic 3: phone, new, money, just, waste, year
  topic 4: dropped, brother, heart, drop, anxiety, phone


In [9]:
df.to_csv("../data/comments_topics.csv", index=False)
print("saved ->", "../data/comments_topics.csv")

saved -> ../data/comments_topics.csv
